# Potential wells: enthalpy (depth) vs entropy (width)

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

A single particle wanders in a **static double-well potential** and is sampled with the
**Metropolis Monte Carlo** algorithm. The two wells differ on purpose:

| | width | depth |
|---|---|---|
| **Well 1** | **wide**  ($2a_1$) | **shallow** ($b_1$) |
| **Well 2** | **narrow** ($2a_2$) | **deep**    ($b_2$) |

The question the simulation answers is: *which well does the particle prefer?*

**The free energy of a flat-bottomed well** of width $w$ and depth $b$ is

$$F = \underbrace{-b}_{\text{enthalpy}}\;\;\underbrace{-\,k_BT\ln w}_{-T\,\Delta S\ \text{(entropy)}}$$

The **depth** is the *enthalpy* term (a deeper well is energetically more favourable), and the
**width** is the *entropy* term (a wider well offers more microstates — more places for the
particle to be). At equilibrium the Boltzmann populations of the two wells obey

$$\frac{P_1}{P_2}=e^{-(F_1-F_2)/k_BT}
      =\underbrace{\frac{w_1}{w_2}}_{\text{entropy}}\;
       \underbrace{e^{(b_1-b_2)/k_BT}}_{\text{enthalpy}} .$$

* At **low temperature**, $e^{(b_1-b_2)/k_BT}$ dominates → the **deeper** well wins (enthalpy).
* At **high temperature**, the exponential $\to 1$ and only $w_1/w_2$ survives → the **wider**
  well wins (entropy).

This competition — enthalpy pulling one way, entropy the other, with temperature as the referee —
is the essence of $\Delta G = \Delta H - T\Delta S$. The simulation lets you *watch* it.


In [ ]:
# --- environment: only matplotlib is third-party (math/random are stdlib) ---
import importlib.util, sys, subprocess
if importlib.util.find_spec("matplotlib") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])

import math, random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rc
from matplotlib.animation import FuncAnimation

rc('animation', html='jshtml')
%matplotlib inline

## 2. The double-well potential

`energy(x)` returns the potential energy at position `x`: $-b_1$ inside well 1, $-b_2$ inside
well 2, and a large positive value (an impassable wall) everywhere else — so the particle is
confined to the two wells. `which_well` reports where the particle currently is, and
`analytic_frac1` is the exact Boltzmann prediction we will check the simulation against.

In [ ]:
BIG = 1.0e99          # a wall: energy outside both wells (effectively +infinity)

# geometry reference point (matches the .py); wells are placed relative to it
startx = 170.0        # left edge reference
l      = 100.0        # gap between the two wells

def well_edges(a1, a2):
    '''(lo1,hi1,lo2,hi2): the x-ranges of well 1 (wide) and well 2 (narrow).'''
    lo1, hi1 = startx - a1,           startx + a1
    lo2, hi2 = startx + a1 + l,       startx + a1 + l + 2*a2
    return lo1, hi1, lo2, hi2

def energy(x, a1, a2, b1, b2):
    lo1, hi1, lo2, hi2 = well_edges(a1, a2)
    if lo1 < x < hi1:  return -b1        # well 1 (wide, shallow)
    if lo2 < x < hi2:  return -b2        # well 2 (narrow, deep)
    return BIG                            # wall

def which_well(x, a1, a2):
    lo1, hi1, lo2, hi2 = well_edges(a1, a2)
    if lo1 < x < hi1:  return 1
    if lo2 < x < hi2:  return 2
    return 0

def analytic_frac1(a1, a2, b1, b2, T, kB):
    '''Exact Boltzmann fraction of time in well 1: (w1/w2) exp((b1-b2)/kT), normalised.'''
    r = (2*a1)/(2*a2) * math.exp((b1 - b2)/(kB*T))
    return r/(1.0 + r)

def draw_potential(ax, a1, a2, b1, b2, top=10.0):
    '''Draw the double-well profile U(x) as a polyline (walls up, floors down).'''
    lo1, hi1, lo2, hi2 = well_edges(a1, a2)
    xs = [lo1-60, lo1, lo1, hi1, hi1, lo2, lo2, hi2, hi2, hi2+60]
    ys = [top,    top, -b1, -b1, top, top, -b2, -b2, top, top]
    ax.plot(xs, ys, color='crimson', lw=2)
    ax.axhline(0, color='0.7', lw=0.8, ls=':')
    ax.set_xlabel('position  x'); ax.set_ylabel('potential energy  U(x)')
    return lo1, hi1, lo2, hi2

## 3. The Metropolis Monte Carlo sampler

One trial move per step: displace the particle by a uniform random step in
$[-\Delta R_{max}, +\Delta R_{max}]$, then **accept** with the Metropolis probability
$\min\!\big(1, e^{-\Delta E/k_BT}\big)$. Crucially, a **rejected** move keeps the current
position, and the well the particle sits in is counted **every** step (accepted or not) — this
residence-time weighting is what makes the recorded populations the true Boltzmann populations.

In [ ]:
def run_mc(a1, a2, b1, b2, T, deltaRmax, nsteps, kB, seed_val, x0=None):
    rng = random.Random(seed_val)
    lo1, hi1, lo2, hi2 = well_edges(a1, a2)
    x = startx if x0 is None else x0          # start in well 1
    E = energy(x, a1, a2, b1, b2)
    nr1 = nr2 = accepted = 0
    x_hist, E_hist, frac_hist = [], [], []
    for step in range(nsteps):
        xnew = x + (2*rng.random() - 1.0)*deltaRmax
        Enew = energy(xnew, a1, a2, b1, b2)
        dE = Enew - E
        if dE < 0.0 or rng.random() < math.exp(-dE/(kB*T)):   # Metropolis (exp underflows to 0)
            x, E = xnew, Enew
            accepted += 1
        # count the CURRENT well every step -> Boltzmann residence weighting
        if   lo1 < x < hi1: nr1 += 1
        elif lo2 < x < hi2: nr2 += 1
        x_hist.append(x); E_hist.append(E)
        frac_hist.append(nr1/(nr1+nr2) if (nr1+nr2) else 0.0)
    return dict(x=x_hist, E=E_hist, frac=frac_hist,
                nr1=nr1, nr2=nr2, accept=accepted/nsteps,
                frac1=nr1/(nr1+nr2) if (nr1+nr2) else 0.0)

## 4. Parameters

**Molecular system and its properties**

| Parameter | Symbol | Meaning |
|---|---|---|
| `a1` | $a_1$ | half-width of **well 1** (its full width is $2a_1$) — the **entropy** of well 1 |
| `a2` | $a_2$ | half-width of **well 2** (full width $2a_2$) — the **entropy** of well 2 |
| `b1` | $b_1$ | depth of **well 1** — the **enthalpy** of well 1 |
| `b2` | $b_2$ | depth of **well 2** — the **enthalpy** of well 2 |
| `l`  | $l$ | gap separating the two wells |
| `startx` | | reference point locating the wells |

Defaults (from the `.py`): well 1 is **wide and shallow** ($2a_1=200$, depth $100$), well 2 is
**narrow and deep** ($2a_2=100$, depth $120$).

**Monte Carlo controls**

| Parameter | Symbol | Meaning |
|---|---|---|
| `Temperature` | $T$ | temperature — the referee between enthalpy and entropy |
| `deltaRmax` | $\Delta R_{max}$ | maximum trial displacement per step |
| `cstboltz` | $k_B$ | Boltzmann constant (the script's teaching value, $8.34\times10^{-3}$) |
| `nsteps` | | number of Monte Carlo steps |
| `Seed` | | RNG seed (reproducibility) |

In [ ]:
# --- molecular system: two wells of different width (entropy) and depth (enthalpy) ---
a1 = 100.0     # well 1 half-width  -> wide  (more entropy)
a2 = 50.0      # well 2 half-width  -> narrow (less entropy)
b1 = 100.0     # well 1 depth       -> shallow (less enthalpy gain)
b2 = 120.0     # well 2 depth       -> deep    (more enthalpy gain)

# --- Monte Carlo controls ---
Temperature = 300.0      # K
deltaRmax   = 500.0      # max trial displacement
cstboltz    = 8.34e-3    # Boltzmann constant (teaching value from the script)
nsteps      = 150000     # MC steps for the main run
Seed        = 100        # RNG seed

for name, (lo, hi) in [("well 1 (wide, shallow)", well_edges(a1,a2)[:2]),
                       ("well 2 (narrow, deep) ", well_edges(a1,a2)[2:])]:
    print(f"{name}: x in ({lo:.0f}, {hi:.0f})  width={hi-lo:.0f}")
print(f"kT at {Temperature:.0f} K = {cstboltz*Temperature:.3f}   (depth gap b2-b1 = {b2-b1:.0f})")

## 5. Run the Monte Carlo

We sample the default system at $T = 300$ K and record the trajectory, the energy, and the
running fraction of time spent in well 1.

In [ ]:
result = run_mc(a1, a2, b1, b2, Temperature, deltaRmax, nsteps, cstboltz, Seed)

pred = analytic_frac1(a1, a2, b1, b2, Temperature, cstboltz)
print(f"steps                 : {nsteps}")
print(f"acceptance ratio      : {result['accept']:.2f}")
print(f"fraction in well 1    : {result['frac1']:.3f}   (analytic Boltzmann: {pred:.3f})")
print(f"fraction in well 2    : {1-result['frac1']:.3f}")
print()
print("At 300 K the depth gap (20) dwarfs kT (2.5), so the DEEP well 2 wins almost completely:")
print("enthalpy dominates. Section 8 turns up the temperature to let entropy compete.")

## 6. Trajectory and convergence

The energy hops between $-b_1$ and $-b_2$ as the particle changes wells; the running
`fraction in well 1` settles onto the equilibrium value (dashed line = analytic Boltzmann).

In [ ]:
fig, (axE, axF) = plt.subplots(1, 2, figsize=(12, 4))

axE.plot(result['E'], lw=0.5, color='steelblue')
axE.axhline(-b1, color='0.6', ls=':'); axE.axhline(-b2, color='0.6', ls=':')
axE.text(nsteps*0.98, -b1, ' -b1 (well 1)', va='center', ha='right', fontsize=9, color='0.4')
axE.text(nsteps*0.98, -b2, ' -b2 (well 2)', va='center', ha='right', fontsize=9, color='0.4')
axE.set_xlabel('MC step'); axE.set_ylabel('energy'); axE.set_title('Energy along the trajectory')

axF.plot(result['frac'], color='seagreen', label='running fraction in well 1')
axF.axhline(pred, color='crimson', ls='--', label=f'analytic Boltzmann = {pred:.3f}')
axF.set_ylim(0, 1); axF.set_xlabel('MC step'); axF.set_ylabel('fraction in well 1')
axF.set_title('Population of well 1 converges'); axF.legend()
plt.tight_layout(); plt.show()

## 7. Where does the particle spend its time?

The red line is the potential; the blue histogram is how often each position was visited. At
300 K the density piles up in the **deep narrow well** — the particle barely samples the wide
shallow one.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
lo1, hi1, lo2, hi2 = draw_potential(ax, a1, a2, b1, b2)
ax.set_title(f'Occupancy at T = {Temperature:.0f} K   '
             f'(well 1: {result["frac1"]:.2f},  well 2: {1-result["frac1"]:.2f})')

ax2 = ax.twinx()
ax2.hist(result['x'], bins=80, color='steelblue', alpha=0.45)
ax2.set_ylabel('visits', color='steelblue'); ax2.tick_params(axis='y', colors='steelblue')
ax.set_zorder(ax2.get_zorder()+1); ax.patch.set_visible(False)
plt.tight_layout(); plt.show()

## 8. The key experiment: population vs temperature

Now sweep the temperature. Each point is an independent MC run; the dashed curve is the exact
Boltzmann law. As $T$ rises the exponential enthalpy factor fades and the **entropic** width
ratio takes over — the wide well 1 gains population.

With the default depths the gap (20) is large, so you need high $T$ before entropy competes —
that is itself the lesson: **a big enthalpy difference needs a lot of thermal energy for entropy
to matter.**

In [ ]:
temps = [200, 400, 700, 1000, 1400, 1900, 2500, 3300, 4300, 5600, 7500, 10000]
scan_steps = 80000

sim  = [run_mc(a1, a2, b1, b2, T, deltaRmax, scan_steps, cstboltz, Seed)['frac1'] for T in temps]
theo = [analytic_frac1(a1, a2, b1, b2, T, cstboltz) for T in temps]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(temps, sim,  'o', color='seagreen', ms=7, label='Monte Carlo')
ax.plot(temps, theo, '--', color='crimson', label='analytic Boltzmann')
ax.axhline(2*a1/(2*a1+2*a2), color='0.6', ls=':',
           label=f'pure-entropy limit  w1/(w1+w2) = {2*a1/(2*a1+2*a2):.2f}')
ax.set_xscale('log')
ax.set_xlabel('temperature  T  (K, log scale)'); ax.set_ylabel('fraction in well 1')
ax.set_ylim(0, 1); ax.set_title('Enthalpy (low T) → entropy (high T)')
ax.legend(); plt.tight_layout(); plt.show()

print("low T  : deep narrow well 2 wins  (enthalpy)")
print("high T : populations approach the width ratio 2:1 -> wide well 1 (entropy)")

## 9. Isolating entropy, and the enthalpy–entropy crossover

Two controlled experiments:

* **Equal depths** ($b_1=b_2$): the enthalpy term cancels, so *only* width matters. The wide
  well holds a $w_1:w_2 = 2:1$ majority at **every** temperature — pure entropy.
* **Small depth gap** ($b_2 = b_1 + 5$): now the deep well wins at low $T$ and the wide well
  wins at high $T$, and the two populations **cross** — $\Delta G = \Delta H - T\Delta S$ in a
  single picture.

In [ ]:
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5))
Ts = [100, 200, 400, 700, 1000, 1500, 2200, 3200, 4600, 7000, 10000]
dm, ns = deltaRmax, 80000

# (A) equal depths -> pure entropy
eq = [run_mc(a1, a2, 100.0, 100.0, T, dm, ns, cstboltz, Seed)['frac1'] for T in Ts]
axA.plot(Ts, eq, 'o-', color='seagreen', label='well 1 (wide)')
axA.axhline(2/3, color='crimson', ls='--', label='2/3  (width ratio, T-independent)')
axA.set_xscale('log'); axA.set_ylim(0, 1); axA.set_xlabel('T (K, log)')
axA.set_ylabel('fraction in well 1'); axA.set_title('Equal depths → pure entropy'); axA.legend()

# (B) small depth gap -> crossover
f1 = [run_mc(a1, a2, 100.0, 105.0, T, dm, ns, cstboltz, Seed)['frac1'] for T in Ts]
axB.plot(Ts, f1,          'o-', color='seagreen', label='well 1 (wide, shallow)')
axB.plot(Ts, [1-f for f in f1], 'o-', color='indigo',  label='well 2 (narrow, deep)')
axB.axhline(0.5, color='0.6', ls=':')
axB.set_xscale('log'); axB.set_ylim(0, 1); axB.set_xlabel('T (K, log)')
axB.set_ylabel('fraction'); axB.set_title('Small depth gap → crossover'); axB.legend()
plt.tight_layout(); plt.show()

## 10. Temperature is not the only knob: depth and width

The temperature scans above held the wells fixed and changed $T$. But the populations depend on
**all** of $b_1, b_2, a_1, a_2$. Here we hold the temperature fixed (at $T = 1000$ K, where the
enthalpy and entropy terms are of comparable size) and instead reshape the wells:

* **Left — change a depth (enthalpy).** Fix both widths and deepen well 2 (increase $b_2$). Each
  extra unit of depth multiplies its Boltzmann weight by $e^{1/k_BT}$, so population drains out of
  well 1 into the deepening well 2.
* **Right — change a width (entropy).** Fix both depths and widen well 1 (increase $a_1$). A wider
  well offers proportionally more microstates, so it steals population — linearly in its width.

In both panels the dots are independent Monte Carlo runs and the dashed line is the exact
Boltzmann law — depth enters **exponentially**, width **linearly**.

In [ ]:
fig, (axD, axW) = plt.subplots(1, 2, figsize=(13, 5))
Tfix, dm, ns = 1000.0, deltaRmax, 80000

# (left) change the DEPTH of well 2 (widths fixed at default a1=100, a2=50)
b2s = [80, 90, 100, 105, 110, 115, 120, 125, 130, 140]
simD  = [run_mc(a1, a2, b1, bb, Tfix, dm, ns, cstboltz, Seed)['frac1'] for bb in b2s]
theoD = [analytic_frac1(a1, a2, b1, bb, Tfix, cstboltz) for bb in b2s]
axD.plot(b2s, simD, 'o', color='indigo', ms=7, label='Monte Carlo')
axD.plot(b2s, theoD, '--', color='crimson', label='analytic Boltzmann')
axD.axvline(b1, color='0.6', ls=':'); axD.text(b1, 0.9, ' $b_2=b_1$', color='0.4', fontsize=9)
axD.set_xlabel('depth of well 2,  $b_2$   (well 1 fixed at $b_1=100$)')
axD.set_ylabel('fraction in well 1'); axD.set_ylim(0, 1)
axD.set_title('Deepen a well → it gains population (enthalpy)'); axD.legend()

# (right) change the WIDTH of well 1 (depths fixed at default b1=100, b2=120)
a1s = [30, 50, 70, 100, 130, 160, 200, 250, 300]
simW  = [run_mc(aa, a2, b1, b2, Tfix, dm, ns, cstboltz, Seed)['frac1'] for aa in a1s]
theoW = [analytic_frac1(aa, a2, b1, b2, Tfix, cstboltz) for aa in a1s]
axW.plot([2*aa for aa in a1s], simW, 'o', color='seagreen', ms=7, label='Monte Carlo')
axW.plot([2*aa for aa in a1s], theoW, '--', color='crimson', label='analytic Boltzmann')
axW.axvline(2*a2, color='0.6', ls=':'); axW.text(2*a2, 0.9, ' width of well 2', color='0.4', fontsize=9)
axW.set_xlabel('width of well 1,  $2a_1$   (well 2 fixed at width $2a_2=100$)')
axW.set_ylabel('fraction in well 1'); axW.set_ylim(0, 1)
axW.set_title('Widen a well → it gains population (entropy)'); axW.legend()
plt.tight_layout(); plt.show()

Both knobs at once: the map below shows the analytic fraction in well 1 over a grid of
well-2 depth ($b_2$) and well-1 width ($2a_1$), still at $T = 1000$ K. Moving **up** (deeper
well 2) drains well 1; moving **right** (wider well 1) fills it. The white star marks the
notebook's default system — the Monte Carlo reproduces every point on this surface.

In [ ]:
import numpy as np
b2_grid = np.linspace(80, 140, 61)          # depth of well 2 (y)
a1_grid = np.linspace(30, 300, 61)          # half-width of well 1 (x)
F = np.array([[analytic_frac1(aa, a2, b1, bb, 1000.0, cstboltz) for aa in a1_grid]
              for bb in b2_grid])

fig, ax = plt.subplots(figsize=(8.5, 5.5))
im = ax.pcolormesh(2*a1_grid, b2_grid, F, cmap='RdYlGn', vmin=0, vmax=1, shading='auto')
cs = ax.contour(2*a1_grid, b2_grid, F, levels=[0.25, 0.5, 0.75], colors='k', linewidths=0.8)
ax.clabel(cs, fmt='%.2f', fontsize=8)
ax.plot(2*a1, b2, marker='*', color='white', ms=18, mec='k', label='default system')
ax.set_xlabel('width of well 1,  $2a_1$'); ax.set_ylabel('depth of well 2,  $b_2$')
ax.set_title('Fraction in well 1 at T = 1000 K  (both knobs together)')
fig.colorbar(im, ax=ax, label='fraction in well 1'); ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

## 11. Watch the particle hop

At the default 300 K the particle is stuck in the deep well and there is nothing to watch, so for
the animation we **warm the system to $T = 1200$ K** — enough thermal energy that it also samples
the wide shallow well about **20 %** of the time. Now you can see the two competing populations:
the disc lingers in the deep narrow well 2 but repeatedly hops across to the wide shallow well 1.

In [ ]:
T_anim, anim_steps = 1200.0, 60000
anim_res = run_mc(a1, a2, b1, b2, T_anim, deltaRmax, anim_steps, cstboltz, Seed)
print(f"animation run at {T_anim:.0f} K : fraction in well 1 = {anim_res['frac1']:.2f}")

stride = max(1, anim_steps // 150)      # cap ~150 frames
frames_x = anim_res['x'][::stride]

fig, ax = plt.subplots(figsize=(9, 4.5))
lo1, hi1, lo2, hi2 = draw_potential(ax, a1, a2, b1, b2)
ax.set_ylim(-max(b1, b2)-20, 25)
ax.set_title(f'T = {T_anim:.0f} K   (well 1: {anim_res["frac1"]:.2f},  well 2: {1-anim_res["frac1"]:.2f})')
disc, = ax.plot([], [], 'o', color='orange', ms=18, mec='k')
label = ax.text(0.02, 0.95, '', transform=ax.transAxes, va='top')

def ypos(x):
    if lo1 < x < hi1: return -b1
    if lo2 < x < hi2: return -b2
    return 0.0

def animate(i):
    x = frames_x[i]
    disc.set_data([x], [ypos(x)])
    label.set_text(f'step {i*stride}')
    return disc, label

anim = FuncAnimation(fig, animate, frames=len(frames_x), interval=60, blit=True)
plt.close(fig)
anim

## 12. Take-home messages

* A well's **depth is enthalpy** and its **width is entropy**: $F = -b - k_BT\ln w$.
* Equilibrium populations follow $\dfrac{P_1}{P_2}=\dfrac{w_1}{w_2}\,e^{(b_1-b_2)/k_BT}$ —
  the product of an **entropic** width ratio and an **enthalpic** Boltzmann factor.
* **Low temperature → enthalpy wins** (the deeper well); **high temperature → entropy wins**
  (the wider well). Temperature is the referee, exactly as in $\Delta G = \Delta H - T\Delta S$.
* **Temperature is not the only knob** (§10): at *fixed* $T$ you can also shift populations by
  reshaping the wells — **deepening** a well pulls population in *exponentially* (enthalpy), while
  **widening** it pulls population in *linearly* (entropy).
* With **equal depths** the wider well keeps a fixed majority at all temperatures — a clean view
  of entropy alone; with a **small depth gap** the populations **cross over** as $T$ increases.
* A large enthalpy difference (like the default gap of 20) overwhelms a modest width ratio until
  the temperature is high enough — which is why the effect is invisible at 300 K here.
* **Methodological note:** proper Metropolis makes *one* trial per step and counts the
  configuration *every* step (a rejected move stays put and is counted again). Discarding
  rejected moves — as the original script did — distorts the populations toward the wider well.